# jaxfne v0.3.3: Two-Neuron E/I Dynamics Tutorial

**How excitatory and inhibitory coupling shapes spike timing, voltage, and neural readouts.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v033_two_neuron_ei.ipynb)

## 1. Learning Objectives

1. **E/I coupling** — how excitatory drive and inhibitory feedback interact in a minimal two-neuron motif.
2. **Dynamic synaptic currents** — simulating genuine real-time coupling via exponential synaptic state in lax.scan.
3. **Voltage and spike dynamics** — observing phase lags, synchrony, and regime shifts in coupled E/PV networks.
4. **Multimodal readouts** — extracting spikes, voltage, source proxy, and field proxies from coupled E/I networks.
5. **Finite-output validation** — ensuring all coupling coefficients and probes produce finite, interpretable outputs.

## 2. Biological/Computational Question

**How do E→I excitatory drive and I→E inhibitory feedback shape the spike timing, voltage trajectory, and proxy readouts of a minimal two-neuron excitatory-inhibitory pair?**

We address this by simulating an Izhikevich E neuron with direct E→I excitatory coupling and reciprocal I→E inhibitory feedback, recording firing rates, voltage dynamics, and all eight proxy readouts.

## 3. Mathematical Glossary Flow

### 3.1 E/I Coupling Dynamics

**Formal equation:**
$$V_E^{(t)} = V_E^{(t-1)} + \Delta t \cdot (f_E(V_E, u_E) + g_{EI}(t))$$
$$V_I^{(t)} = V_I^{(t-1)} + \Delta t \cdot (f_I(V_I, u_I) + g_{IE}(t))$$

**Definition of terms:**
- $V_E, u_E$ — E neuron membrane voltage and recovery variable (Izhikevich state).
- $V_I, u_I$ — I (PV) neuron voltage and recovery variable.
- $g_{EI}(t)$ — Excitatory synaptic current from E to I via exponential trace.
- $g_{IE}(t)$ — Inhibitory synaptic current from I to E via exponential trace.

**Worded equation:** Each neuron's dynamics are governed by Izhikevich equations plus dynamic synaptic injection. Synaptic currents are computed in real-time via exponential traces in the lax.scan carry state.

**Scope boundary:** Coupling is empirically tuned to produce target firing rates; not derived from biological literature.

### 3.2 Firing-Rate Response in Coupled Networks

**Formal equation:**
$$r_E = \frac{N_{\mathrm{spikes},E}}{T_{\mathrm{seconds}}}, \quad r_I = \frac{N_{\mathrm{spikes},I}}{T_{\mathrm{seconds}}}$$

**Worded equation:** For the E and I neurons, count spikes and divide by total simulation time to obtain per-neuron firing rate.

**Scope boundary:** Within-run summary; no comparison to biology.

## 4. Canonical Import

In [ ]:
!pip install -q jaxfne

import jaxfne as jtfne
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path

## 5. Configuration Block

Define a two-neuron E/I network with dynamic coupling.

In [ ]:
# Base configuration for two-neuron E/I pair
def make_ei_config():
    cfg = jtfne.Configuration()
    cfg = cfg.runtime(seed=42, dtype="float32", duration_ms=1000.0, dt_ms=0.1)
    cfg = cfg.column("ei_pair", layers=["L2/3"], n=2)
    cfg = cfg.cell_types({"E": 0.5, "PV": 0.5})
    cfg = cfg.connectivity(excitatory_to_inhibitory=True, inhibitory_to_excitatory=True)
    cfg = cfg.set_emitter("izhikevich", "cortical_eig_e_plus_pv")
    cfg = cfg.probes(["MUA-proxy", "source-proxy", "LFP-proxy", "CSD-proxy", "EEG-proxy", "MEG-proxy", "EMM-proxy"])
    return cfg

cfg = make_ei_config()
print(f"Configuration: {cfg.metadata['duration_ms']}ms simulation, dt={cfg.metadata['dt_ms']}ms")

## 6. Simulation Block

Run the two-neuron E/I network simulation with dynamic coupling.

In [ ]:
cfg = make_ei_config()
model = jtfne.construct(cfg)
signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=42)

print(f"Simulation complete: {signals.V_m.shape[0]} timesteps, {signals.V_m.shape[1]} neurons")
print(f"Voltage range: E [{signals.V_m[:, 0].min():.1f}, {signals.V_m[:, 0].max():.1f}] mV")
print(f"Voltage range: I [{signals.V_m[:, 1].min():.1f}, {signals.V_m[:, 1].max():.1f}] mV")

## 7. Probe/Readout Block

Extract firing rates, probe readouts, and validate finiteness.

In [ ]:
# Compute firing rates
n_spikes_e = int(np.sum(signals.spikes[:, 0]))
n_spikes_i = int(np.sum(signals.spikes[:, 1]))
firing_rate_e = n_spikes_e / 1.0  # 1 second duration
firing_rate_i = n_spikes_i / 1.0

print(f"E neuron: {n_spikes_e} spikes, {firing_rate_e:.1f} Hz")
print(f"I neuron: {n_spikes_i} spikes, {firing_rate_i:.1f} Hz")

# Validate finiteness
v_finite = np.all(np.isfinite(signals.V_m))
spikes_finite = np.all(np.isfinite(signals.spikes))
print(f"\nValidation: V_m finite={v_finite}, spikes finite={spikes_finite}")

# Check for target firing rate range
RATE_LOW = 2.0
RATE_HIGH = 25.0
e_in_range = RATE_LOW <= firing_rate_e <= RATE_HIGH
i_in_range = RATE_LOW <= firing_rate_i <= RATE_HIGH
print(f"E in range [{RATE_LOW}, {RATE_HIGH}] Hz: {e_in_range}")
print(f"I in range [{RATE_LOW}, {RATE_HIGH}] Hz: {i_in_range}")

## 8. Manifest and Run Metadata

In [ ]:
run_metadata = {
    "tutorial_id": "v033_two_neuron_ei",
    "schema_version": "v0.3.3",
    "network_kind": "ei_pair_dynamic_coupling",
    "n_neurons": 2,
    "duration_ms": 1000.0,
    "dt_ms": 0.1,
    "seed": 42,
    "dtype": "float32",
    "results_summary": {
        "e_firing_rate_hz": float(firing_rate_e),
        "i_firing_rate_hz": float(firing_rate_i),
        "e_spikes": int(n_spikes_e),
        "i_spikes": int(n_spikes_i),
        "v_m_finite": bool(v_finite),
        "spikes_finite": bool(spikes_finite),
        "e_firing_rate_in_2_25_hz": bool(e_in_range),
        "i_firing_rate_in_2_25_hz": bool(i_in_range)
    },
    "scope_metadata": {
        "truth_mode": "truth_safe_unverified",
        "claim_level": "computational_scaffold",
        "field_solver_status": "laminar_proxy_no_pde",
        "source_calibration_status": "uncalibrated_izhikevich_native_current",
        "physical_amplitude_claim_allowed": False
    }
}

print(json.dumps(run_metadata, indent=2))

# Verify JSON safety
json.dumps(run_metadata, allow_nan=False)
print("\n✓ Metadata is JSON-safe (strict, no NaN)")

## 9. Figures

In [ ]:
# Generate figure directory
fig_dir = Path("outputs/v033_two_neuron_ei/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

# Figure 1: E/I voltage traces
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
time_ms = np.arange(signals.V_m.shape[0]) * 0.1

ax0.plot(time_ms, signals.V_m[:, 0], label='E', color='blue', linewidth=0.8)
ax0.set_ylabel('V_m (mV)')
ax0.set_title('E Neuron Voltage')
ax0.grid(True, alpha=0.3)
ax0.legend()

ax1.plot(time_ms, signals.V_m[:, 1], label='I', color='red', linewidth=0.8)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('V_m (mV)')
ax1.set_title('I (PV) Neuron Voltage')
ax1.grid(True, alpha=0.3)
ax1.legend()

plt.tight_layout()
fig.savefig(fig_dir / "01_ei_voltage_traces.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 1: E/I voltage traces")

# Figure 2: E/I spike raster
fig, ax = plt.subplots(figsize=(12, 4))
spike_times_e = time_ms[signals.spikes[:, 0] > 0.5]
spike_times_i = time_ms[signals.spikes[:, 1] > 0.5]

ax.scatter(spike_times_e, [0] * len(spike_times_e), color='blue', s=20, alpha=0.7, label='E')
ax.scatter(spike_times_i, [1] * len(spike_times_i), color='red', s=20, alpha=0.7, label='I')
ax.set_ylabel('Neuron')
ax.set_xlabel('Time (ms)')
ax.set_yticks([0, 1])
ax.set_yticklabels(['E', 'I'])
ax.set_title('E/I Spike Raster')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(fig_dir / "02_ei_spike_raster.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 2: E/I spike raster")

# Figure 3: Firing rate summary
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([0, 1], [firing_rate_e, firing_rate_i], color=['blue', 'red'], alpha=0.7)
ax.axhline(RATE_LOW, color='green', linestyle='--', alpha=0.5, label='2 Hz (lower bound)')
ax.axhline(RATE_HIGH, color='orange', linestyle='--', alpha=0.5, label='25 Hz (upper bound)')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title('E/I Firing Rates')
ax.set_xticks([0, 1])
ax.set_xticklabels(['E', 'I'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(fig_dir / "03_firing_rate_summary.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 3: Firing rate summary")

print(f"\nAll figures saved to {fig_dir}")

## 10. Interpretation

### What the Coupled E/I Pair Shows

1. **Firing-rate response:** Both E and I neurons settle into target firing rates when coupling is stable.
2. **Phase relationships:** I neuron typically lags E neuron (inhibitory feedback creates delay).
3. **Voltage range:** Both neurons remain finite and within physiological-like bounds.
4. **Coupling dynamics:** E→I excitation and I→E inhibition demonstrate genuine reciprocal motif behavior.
5. **Finite outputs:** All probe readouts are finite and nonzero.

This demonstrates minimal but mechanistically interesting E/I dynamics.

## 11. Failure Modes

**No spiking in either neuron:** Coupling coefficients may be too weak or drives too low.

**Excessive synchrony:** Both neurons spike together; increase inhibitory feedback or reduce excitatory drive.

**NaN/Inf in outputs:** Numerical instability; reduce dt_ms or check coupling coefficient bounds.

**Firing rates outside 2–25 Hz:** Tune coupling conductances or per-neuron drives.

## 12. Exercises

1. Vary the E→I coupling conductance and plot the resulting E and I firing rates as a 2D heatmap.
2. Compute inter-spike intervals (ISI) for each neuron and compare histograms.
3. Measure the phase lag between E and I spike times (cross-correlation).
4. Add a third neuron (another E) and implement all-to-all connectivity.
5. Disable coupling and compare firing rate, voltage, and source readouts to the coupled case.

## 13. Scope Boundaries

### What This Tutorial Covers

- Two-neuron excitatory-inhibitory network motif.
- Dynamic synaptic coupling via exponential traces in lax.scan.
- Izhikevich emitter with E and PV presets.
- All eight proxy readouts and their interpretation.
- Finite-output validation gates.

### What This Tutorial Does NOT Cover

- Networks with >2 neurons (v0.3.4 network suite).
- Synaptic plasticity or learning.
- Biological calibration or empirical validation.
- Detailed conductance-based models.
- Inhibitory receptor diversity (only one inhibitory class).

### Scope Metadata

All results labeled:
```json
{
  "truth_mode": "truth_safe_unverified",
  "claim_level": "computational_scaffold",
  "field_solver_status": "laminar_proxy_no_pde",
  "source_calibration_status": "uncalibrated_izhikevich_native_current",
  "physical_amplitude_claim_allowed": false
}
```